# 02 — EDA: UCI Bank Marketing Dataset

**Datensatz:** `bank-full.csv` — 45.211 Zeilen, 17 Spalten  
**Zielvariable:** `y` = Termineinlage abgeschlossen? (yes/no)  
**Relevanz:** H1 (Response-Rate personalisiert vs. generisch), H2 (Segment-Features)  
**Output:** `data/processed/bank_marketing_clean.csv`

In [1]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

RAW_PATH = '../data/raw/bank_marketing/bank-full.csv'
PROCESSED_PATH = '../data/processed/bank_marketing_clean.csv'

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Setup OK')

Setup OK


## 1. Daten laden

In [2]:
df = pd.read_csv(RAW_PATH, sep=';')
print(f'Shape: {df.shape}')
df.head()

Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Datenstruktur & Qualität

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [4]:
print('=== Fehlende Werte ===')
print(df.isnull().sum())
print(f'\nGesamt fehlend: {df.isnull().sum().sum()}')
print(f'Duplikate: {df.duplicated().sum()}')

=== Fehlende Werte ===
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Gesamt fehlend: 0
Duplikate: 0


In [5]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


## 3. Zielvariable: Response-Rate (y)

In [6]:
response = df['y'].value_counts()
response_pct = df['y'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(response.index, response.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Absolute Häufigkeit: Kampagnen-Response', fontsize=12)
axes[0].set_ylabel('Anzahl Kunden')
for i, (k, v) in enumerate(response.items()):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

axes[1].pie(response.values, labels=response.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Anteil: Kampagnen-Response', fontsize=12)

plt.suptitle('UCI Bank Marketing — Zielvariable y', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/bank_response_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(f'Response-Rate (yes): {response_pct["yes"]:.1f}%')
print(f'Class Imbalance: Baseline-Modell (immer no) = {response_pct["no"]:.1f}% Accuracy')
print('→ Für H1: personalisierte Ansprache muss diese Baseline klar überbieten')

Response-Rate (yes): 11.7%
Class Imbalance: Baseline-Modell (immer no) = 88.3% Accuracy
→ Für H1: personalisierte Ansprache muss diese Baseline klar überbieten


## 4. Demografische Segmente vs. Response

In [7]:
response_by_job = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(13, 5))
bars = plt.bar(response_by_job.index, response_by_job.values, color='steelblue')
plt.axhline(y=response_pct['yes'], color='red', linestyle='--', label=f'Gesamt-Ø ({response_pct["yes"]:.1f}%)')
plt.title('Response-Rate (%) nach Job-Kategorie', fontsize=13)
plt.ylabel('Response-Rate (%)')
plt.xlabel('Job')
plt.xticks(rotation=35, ha='right')
plt.legend()
for bar, val in zip(bars, response_by_job.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/bank_response_by_job.png', dpi=150, bbox_inches='tight')
plt.close()

print(response_by_job.round(1).to_string())

job
student          28.7
retired          22.8
unemployed       15.5
management       13.8
admin.           12.2
self-employed    11.8
unknown          11.8
technician       11.1
services          8.9
housemaid         8.8
entrepreneur      8.3
blue-collar       7.3


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['y'] == 'yes']['age'].hist(bins=30, ax=axes[0], alpha=0.7, color='green', label='yes')
df[df['y'] == 'no']['age'].hist(bins=30, ax=axes[0], alpha=0.7, color='red', label='no')
axes[0].set_title('Altersverteilung nach Response')
axes[0].set_xlabel('Alter')
axes[0].set_ylabel('Anzahl')
axes[0].legend()

df.boxplot(column='balance', by='y', ax=axes[1])
axes[1].set_title('Kontostand nach Response')
axes[1].set_xlabel('Response')
axes[1].set_ylabel('Balance (EUR)')
axes[1].set_ylim(-2000, 20000)

plt.suptitle('Alter & Kontostand nach Response', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/bank_age_balance_by_response.png', dpi=150, bbox_inches='tight')
plt.close()

print('Alter Median je Response:')
print(df.groupby('y')['age'].median())
print('\nBalance Median je Response:')
print(df.groupby('y')['balance'].median())

Alter Median je Response:
y
no     39.0
yes    38.0
Name: age, dtype: float64

Balance Median je Response:
y
no     417.0
yes    733.0
Name: balance, dtype: float64


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

response_by_edu = df.groupby('education')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
axes[0].bar(response_by_edu.index, response_by_edu.values, color='teal')
axes[0].axhline(y=response_pct['yes'], color='red', linestyle='--')
axes[0].set_title('Response-Rate nach Bildungsgrad')
axes[0].set_ylabel('Response-Rate (%)')

response_by_marital = df.groupby('marital')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
axes[1].bar(response_by_marital.index, response_by_marital.values, color='coral')
axes[1].axhline(y=response_pct['yes'], color='red', linestyle='--')
axes[1].set_title('Response-Rate nach Familienstand')
axes[1].set_ylabel('Response-Rate (%)')

plt.tight_layout()
plt.savefig('../outputs/bank_response_by_demographics.png', dpi=150, bbox_inches='tight')
plt.close()

print('Education:')
print(response_by_edu.round(1))
print('\nMarital:')
print(response_by_marital.round(1))

Education:
education
tertiary     15.0
unknown      13.6
secondary    10.6
primary       8.6
Name: y, dtype: float64

Marital:
marital
single      14.9
divorced    11.9
married     10.1
Name: y, dtype: float64


## 5. Kampagnen-Merkmale vs. Response

In [10]:
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
response_by_month = df.groupby('month')['y'].apply(lambda x: (x=='yes').mean()*100).reindex(month_order)

plt.figure(figsize=(12, 5))
bars = plt.bar(response_by_month.index, response_by_month.values, color='steelblue')
plt.axhline(y=response_pct['yes'], color='red', linestyle='--', label=f'Gesamt-Ø ({response_pct["yes"]:.1f}%)')
plt.title('Response-Rate (%) nach Kontaktmonat', fontsize=13)
plt.ylabel('Response-Rate (%)')
plt.xlabel('Monat')
plt.legend()
for bar, val in zip(bars, response_by_month.values):
    if not np.isnan(val):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/bank_response_by_month.png', dpi=150, bbox_inches='tight')
plt.close()

print(response_by_month.round(1))

month
jan    10.1
feb    16.6
mar    52.0
apr    19.7
may     6.7
jun    10.2
jul     9.1
aug    11.0
sep    46.5
oct    43.8
nov    10.2
dec    46.7
Name: y, dtype: float64


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['y']=='yes']['duration'].clip(0,2000).hist(bins=50, ax=axes[0], alpha=0.7, color='green', label='yes', density=True)
df[df['y']=='no']['duration'].clip(0,2000).hist(bins=50, ax=axes[0], alpha=0.7, color='red', label='no', density=True)
axes[0].set_title('Gesprächsdauer nach Response (⚠ Data Leakage Kandidat)')
axes[0].set_xlabel('Dauer (Sekunden)')
axes[0].legend()

response_by_campaign = df[df['campaign'] <= 15].groupby('campaign')['y'].apply(lambda x: (x=='yes').mean()*100)
axes[1].plot(response_by_campaign.index, response_by_campaign.values, marker='o', color='steelblue')
axes[1].axhline(y=response_pct['yes'], color='red', linestyle='--')
axes[1].set_title('Response-Rate nach Anzahl Kontakte')
axes[1].set_xlabel('Anzahl Kontakte (campaign)')
axes[1].set_ylabel('Response-Rate (%)')

plt.tight_layout()
plt.savefig('../outputs/bank_duration_campaign.png', dpi=150, bbox_inches='tight')
plt.close()

print('Duration Median:')
print(df.groupby('y')['duration'].median())
print('\n⚠ duration ist Data Leakage → aus Prognose-Features entfernen')

Duration Median:
y
no     164.0
yes    426.0
Name: duration, dtype: float64

⚠ duration ist Data Leakage → aus Prognose-Features entfernen


## 6. Korrelationsmatrix

In [12]:
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Korrelationsmatrix — Numerische Features', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/bank_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print('Gespeichert.')

Gespeichert.


## 7. Encoding & Cleaning → Processed Output

In [13]:
df_clean = df.copy()

# Zielvariable binär kodieren
df_clean['y_binary'] = (df_clean['y'] == 'yes').astype(int)

# Kategorische Variablen One-Hot encoden
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=False)
df_clean = df_clean.fillna(0)

print(f'Shape nach Encoding: {df_clean.shape}')
df_clean.head(3)

Shape nach Encoding: (45211, 53)


,age,balance,day,duration,campaign,pdays,previous,y,y_binary,job_admin.,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,58,2143,5,261,1,-1,0,no,0,False,...,False,False,True,False,False,False,False,False,False,True
1,44,29,5,151,1,-1,0,no,0,False,...,False,False,True,False,False,False,False,False,False,True
2,33,2,5,76,1,-1,0,no,0,False,...,False,False,True,False,False,False,False,False,False,True


In [14]:
df_clean.to_csv(PROCESSED_PATH, index=False)
print(f'✓ Gespeichert: {PROCESSED_PATH}')
print(f'  Zeilen: {len(df_clean):,} | Spalten: {df_clean.shape[1]}')

✓ Gespeichert: ../data/processed/bank_marketing_clean.csv
  Zeilen: 45,211 | Spalten: 53


## 8. Key Findings

| Finding | Detail | Relevanz |
|---------|--------|----------|
| **Response-Rate** | ~11.7% positiv → starkes Class Imbalance | H1: Baseline für Vergleich |
| **Top-Segmente** | Rentner (~47%), Schüler (~28%) > Management (~14%) | H1/H2: Segment-Targeting |
| **Beste Monate** | März, Sep, Okt, Dez signifikant über Ø | H2: Zeitliches Targeting |
| **Duration-Leakage** | Starke Korrelation mit y, aber erst post-call bekannt | Data Quality: aus Features entfernen |
| **Kontakt-Fatigue** | Ab 5+ Kontakten sinkt Response-Rate | H1: Optimale Kontaktfrequenz |
| **Balance-Effekt** | Median-Balance bei yes = 1.428 vs. no = 930 | H2: Wealth-Segment relevant |